In [1]:
import torch
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from lume_torch.variables import TorchScalarVariable, DistributionVariable
from lume_torch.models.gp_model import GPModel

/Users/smiskov/miniconda3/envs/lume-torch-latest/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Multi-output example

In [2]:
torch.manual_seed(0)

# Create training data, 1 input, 3 outputs
train_x = torch.rand(5, 1)
train_y = torch.stack(
    (
        torch.sin(train_x * (2 * torch.pi)) + 0.1 * torch.randn(1),
        torch.cos(train_x * (2 * torch.pi)) + 0.1 * torch.randn(1),
        torch.sin(2 * train_x * (2 * torch.pi)) + 0.1 * torch.randn(1),
    ),
    dim=-1,
).squeeze(1)


# Initialize the GP model
model = SingleTaskGP(train_x.to(dtype=torch.double), train_y.to(dtype=torch.double))

# Fit the model
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll)

# Derive posterior mean and variance
model.eval()
test_x = torch.linspace(0, 1, 10).reshape(-1, 1).to(dtype=torch.double)
posterior = model.posterior(test_x)

# Derive the posterior mean and variance for each output
mean = posterior.mean
variance = posterior.variance
print("Posterior Mean for each output:")
print(mean)
print("\nPosterior Variance for each output:")
print(variance)

Posterior Mean for each output:
tensor([[ 0.2057,  1.0192,  0.3797],
        [ 0.6948,  0.7911,  1.0457],
        [ 1.0173,  0.2163,  0.2572],
        [ 0.9157, -0.4811, -0.6407],
        [ 0.3962, -0.9256, -0.1664],
        [-0.2683, -0.8640,  0.0784],
        [-0.7624, -0.3809, -0.0426],
        [-0.9379,  0.1794, -0.1595],
        [-0.8441,  0.5149, -0.0208],
        [-0.6245,  0.5696,  0.0715]], dtype=torch.float64,
       grad_fn=<CloneBackward0>)

Posterior Variance for each output:
tensor([[0.0294, 0.0322, 0.2155],
        [0.0015, 0.0014, 0.0026],
        [0.0033, 0.0035, 0.1020],
        [0.0028, 0.0029, 0.0347],
        [0.0028, 0.0030, 0.1173],
        [0.0059, 0.0066, 0.1769],
        [0.0115, 0.0143, 0.3564],
        [0.0034, 0.0033, 0.0091],
        [0.0746, 0.0872, 0.4328],
        [0.2720, 0.2989, 0.5247]], dtype=torch.float64,
       grad_fn=<CloneBackward0>)


## LUME-Model import

In [3]:
# Define input variables
input_variables = [TorchScalarVariable(name="x")]

# Define output variables
output_variables = [
    DistributionVariable(name="output1"),
    DistributionVariable(name="output2"),
    DistributionVariable(name="output3"),
]

# Create lume_torch instance
gp_lume_torch = GPModel(
    model=model, input_variables=input_variables, output_variables=output_variables
)

### Evaluate model and run methods

In [4]:
input_dict = {"x": test_x.to(dtype=torch.double)}

In [5]:
input_dict

{'x': tensor([[0.0000],
         [0.1111],
         [0.2222],
         [0.3333],
         [0.4444],
         [0.5556],
         [0.6667],
         [0.7778],
         [0.8889],
         [1.0000]], dtype=torch.float64)}

In [6]:
# Evaluate function returns a dictionary mapping each output to a torch.distributions.Distribution
output_dict = gp_lume_torch.evaluate(input_dict)

In [7]:
output_dict

{'output1': MultivariateNormal(loc: torch.Size([10]), covariance_matrix: torch.Size([10, 10])),
 'output2': MultivariateNormal(loc: torch.Size([10]), covariance_matrix: torch.Size([10, 10])),
 'output3': MultivariateNormal(loc: torch.Size([10]), covariance_matrix: torch.Size([10, 10]))}

In [8]:
test_prob = output_dict["output1"].sample(torch.Size([2]))

In [9]:
print("Posterior mean:", output_dict["output1"].mean)
print("Posterior Variance ", output_dict["output1"].variance)
print("Log Likelihood", output_dict["output1"].log_prob(test_prob))
print("Rsample ", output_dict["output1"].rsample(torch.Size([3])))

Posterior mean: tensor([ 0.2057,  0.6948,  1.0173,  0.9157,  0.3962, -0.2683, -0.7624, -0.9379,
        -0.8441, -0.6245], dtype=torch.float64, grad_fn=<ExpandBackward0>)
Posterior Variance  tensor([0.0294, 0.0015, 0.0033, 0.0028, 0.0028, 0.0059, 0.0115, 0.0034, 0.0746,
        0.2720], dtype=torch.float64, grad_fn=<ExpandBackward0>)
Log Likelihood tensor([17.3171, 15.8412], dtype=torch.float64, grad_fn=<SubBackward0>)
Rsample  tensor([[ 0.1544,  0.7000,  0.9810,  0.8623,  0.3631, -0.3833, -0.9663, -0.9788,
         -0.4930,  0.0778],
        [ 0.3099,  0.6627,  0.9307,  0.8558,  0.4026, -0.1392, -0.5748, -0.9376,
         -1.1807, -1.1273],
        [ 0.4233,  0.6754,  0.9243,  0.8847,  0.4144, -0.3218, -0.9260, -1.0431,
         -0.7037, -0.3066]], dtype=torch.float64, grad_fn=<ViewBackward0>)


### Outputs with original model

In [10]:
print("Posterior mean:", posterior.mean[:, 0])
print("Posterior Variance ", posterior.variance[:, 0])

Posterior mean: tensor([ 0.2057,  0.6948,  1.0173,  0.9157,  0.3962, -0.2683, -0.7624, -0.9379,
        -0.8441, -0.6245], dtype=torch.float64, grad_fn=<SelectBackward0>)
Posterior Variance  tensor([0.0294, 0.0015, 0.0033, 0.0028, 0.0028, 0.0059, 0.0115, 0.0034, 0.0746,
        0.2720], dtype=torch.float64, grad_fn=<SelectBackward0>)


# A 3D Rosenbrock example for GPModel class

## Create and train a GP

In [11]:
# Define the 3D Rosenbrock function
def rosenbrock(X):
    x1, x2, x3 = X[..., 0], X[..., 1], X[..., 2]
    return (
        (1 - x1) ** 2
        + 100 * (x2 - x1**2) ** 2
        + (1 - x2) ** 2
        + 100 * (x3 - x2**2) ** 2
    )

In [12]:
# Generate training data
train_x = torch.rand(20, 3) * 4 - 2  # 20 points in 3D space, scaled to [-2, 2]
train_y = rosenbrock(train_x).unsqueeze(-1)  # Compute the Rosenbrock function values

# Define the GP model
gp_model = SingleTaskGP(train_x.to(dtype=torch.double), train_y.to(dtype=torch.double))

# Fit the model
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)
fit_gpytorch_mll(mll)

# Evaluate the model on test data
test_x = torch.rand(10, 3) * 4 - 2  # 10 new points in 3D space
gp_model.eval()
posterior = gp_model.posterior(test_x)

# Get the mean and variance of the posterior
mean = posterior.mean
variance = posterior.variance

print("Posterior mean: ", mean)
print("Posterior variance: ", variance)

Posterior mean:  tensor([[ 606.3839],
        [ 685.3384],
        [1196.5879],
        [1049.7149],
        [ 258.3421],
        [1224.6845],
        [ 926.0569],
        [ 801.6735],
        [ 334.0830],
        [ 529.1804]], dtype=torch.float64, grad_fn=<UnsqueezeBackward0>)
Posterior variance:  tensor([[ 28350.0395],
        [246126.2304],
        [129956.1858],
        [ 73671.8775],
        [ 27067.5135],
        [154205.9664],
        [180241.9590],
        [206996.4181],
        [ 44488.5649],
        [ 73535.2868]], dtype=torch.float64, grad_fn=<UnsqueezeBackward0>)


/Users/smiskov/miniconda3/envs/lume-torch-latest/lib/python3.14/site-packages/botorch/models/utils/assorted.py:276: InputDataWarning: Data (input features) is not contained to the unit cube. Please consider min-max scaling the input data.
  check_min_max_scaling(


## LUME-Model import

In [13]:
# Define input variables
input_variables = [
    TorchScalarVariable(name="x1"),
    TorchScalarVariable(name="x2"),
    TorchScalarVariable(name="x3"),
]

# Define output variables
output_variables = [DistributionVariable(name="output1")]

# Create lume_torch instance
gp_lume_torch = GPModel(
    model=gp_model, input_variables=input_variables, output_variables=output_variables
)

#### Evaluate model and run methods

In [14]:
input_x = torch.rand(3, 3) * 4 - 2  # 3 new points in 3D space
input_dict = {
    "x1": input_x[:, 0].unsqueeze(1).to(dtype=torch.double),
    "x2": input_x[:, 1].unsqueeze(1).to(dtype=torch.double),
    "x3": input_x[:, 2].unsqueeze(1).to(dtype=torch.double),
}

In [15]:
# Evaluate function returns a dictionary mapping each output to a torch.distributions.Distribution
lume_dist = gp_lume_torch.evaluate(input_dict)["output1"]

In [16]:
rand_test = torch.rand(1, 3)

In [17]:
print("Posterior mean:", lume_dist.mean)
print("Posterior Variance ", lume_dist.variance)
print("Log Likelihood", lume_dist.log_prob(rand_test))
print("Rsample ", lume_dist.rsample(torch.Size([3])))

Posterior mean: tensor([ 144.1534,  750.6190, 1551.5102], dtype=torch.float64,
       grad_fn=<ExpandBackward0>)
Posterior Variance  tensor([107860.9079, 105648.6793,  77374.0175], dtype=torch.float64,
       grad_fn=<ExpandBackward0>)
Log Likelihood tensor([-35.7136], dtype=torch.float64, grad_fn=<SubBackward0>)
Rsample  tensor([[ 225.9312,  858.3771, 2018.9276],
        [ 315.6748, 1014.5847, 1271.6913],
        [ 368.0957, 1014.4358, 2090.0456]], dtype=torch.float64,
       grad_fn=<ViewBackward0>)


### Outputs with original model

In [18]:
posterior = gp_model.posterior(input_x)
botorch_dist = posterior.distribution

In [19]:
print("Posterior mean:", botorch_dist.mean)
print("Posterior Variance ", botorch_dist.variance)
print("Log Likelihood", botorch_dist.log_prob(rand_test))
print("Rsample ", botorch_dist.rsample(torch.Size([3])))

Posterior mean: tensor([ 144.1534,  750.6190, 1551.5102], dtype=torch.float64,
       grad_fn=<AddBackward0>)
Posterior Variance  tensor([107860.9079, 105648.6793,  77374.0175], dtype=torch.float64,
       grad_fn=<ExpandBackward0>)
Log Likelihood tensor([-35.7136], dtype=torch.float64, grad_fn=<SubBackward0>)
Rsample  tensor([[  76.0090,  322.3840, 1341.8768],
        [ 129.2442,  821.3236, 1278.1483],
        [-149.8472,  711.9353, 1303.4597]], dtype=torch.float64,
       grad_fn=<ViewBackward0>)
